<a href="https://colab.research.google.com/github/sultanjacob/Applied-Machine-Learning/blob/main/Phase_11_K_Nearest_Neighbour/11_kNN_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 11: Classification via k-Nearest Neighbors (k-NN)
## Step 1: Loading the Compressed Data & Splitting the Environment

In Phase 10, we aggressively reduced our 590-sensor manufacturing dataset down to 162 mathematically independent components, preserving 95% of the vital business signal while eliminating the "Curse of Dimensionality."

Our first step is to load this optimized artifact (`secom_pca_95variance_ready.csv`) and split it into Training and Testing sets. Because predictive maintenance datasets are notoriously imbalanced (factory machines pass inspection far more often than they fail), we must use a **Stratified Split**. This guarantees that the rare anomalies are distributed evenly between our training and testing environments, preventing the model from learning a skewed representation of reality.

In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

print("⏳ Loading PCA-compressed factory data...")

# 1. Load the artifact exported from Phase 10
dataset_path = 'secom_pca_95variance_ready.csv'
df = pd.read_csv(dataset_path)

# 2. Separate the Principal Components from the Target Label
X = df.drop(columns=['Target'])
y = df['Target']

# 3. Execute a Stratified Train/Test Split (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# 4. Verify the distribution of anomalies
print("✅ Data Loaded and Split Successfully!\n")
print("📊 Training Environment:")
print(f"Total Batches: {len(y_train)}")
print(f"Normal Passes (-1): {sum(y_train == -1)}")
print(f"Anomalies / Fails (1): {sum(y_train == 1)}")

print("\n🧪 Testing Environment:")
print(f"Total Batches: {len(y_test)}")
print(f"Normal Passes (-1): {sum(y_test == -1)}")
print(f"Anomalies / Fails (1): {sum(y_test == 1)}")

⏳ Loading PCA-compressed factory data...
✅ Data Loaded and Split Successfully!

📊 Training Environment:
Total Batches: 1253
Normal Passes (-1): 1170
Anomalies / Fails (1): 83

🧪 Testing Environment:
Total Batches: 314
Normal Passes (-1): 293
Anomalies / Fails (1): 21


## Step 2: Instantiating and Training the k-NN Engine

With our anomalies perfectly stratified, we can build the k-Nearest Neighbors classifier.

Because k-NN is purely distance-based and we have already mathematically condensed our features using PCA, this model will be incredibly lightweight and fast. This minimal computational footprint is exactly what you want when packaging a predictive engine to run efficiently as the backend for an iOS or Android app later.

We will start by setting `n_neighbors=5`. This means for every new batch, the algorithm will look at the 5 closest historical batches in our 162-dimensional space and take a majority vote to decide if the new batch is a Pass or a Fail.

In [2]:
from sklearn.neighbors import KNeighborsClassifier
import time

print("⚙️ Initializing k-NN Engine (k=5)...")

# 1. Initialize the classifier with 5 neighbors
knn_baseline = KNeighborsClassifier(n_neighbors=5)

# 2. Train the model and track the execution time
start_time = time.time()
knn_baseline.fit(X_train, y_train)
training_time = time.time() - start_time

# 3. Generate predictions for the unseen test environment
y_pred = knn_baseline.predict(X_test)

print(f"✅ Model trained in {training_time:.4f} seconds.")
print("🔮 Predictions generated for the 314 test batches.")

⚙️ Initializing k-NN Engine (k=5)...
✅ Model trained in 0.0446 seconds.
🔮 Predictions generated for the 314 test batches.
